# 06b — Train credit default models (Lending Club)

Train and evaluate **Logistic Regression** (with `StandardScaler`) and **XGBoost** (unscaled, with **`scale_pos_weight`** from `y_train` for imbalance) on the pre-built dataset. **No preprocessing** — load, split, scale (LR only), train, evaluate, persist.

Model selection uses **recall → ROC AUC → F1** (not raw accuracy).

**Input:** `datasets/processed/lendingclub_real_training_ready.csv`  
**Artifacts:** `backend/models/` (`joblib`)

## 1. Load data

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for directory in (cwd, *cwd.parents):
        if (directory / "backend").is_dir() and (directory / "ml").is_dir():
            return directory
    return cwd


def resolve_training_csv(processed_dir: Path) -> Path:
    canonical = processed_dir / "lendingclub_real_training_ready.csv"
    if canonical.is_file():
        return canonical.resolve()
    if not processed_dir.is_dir():
        raise FileNotFoundError(f"Missing directory: {processed_dir.resolve()}")
    hits = sorted(processed_dir.glob("*real*training*ready*.csv"))
    if not hits:
        listing = sorted(p.name for p in processed_dir.glob("*.csv"))
        raise FileNotFoundError(
            f"Expected lendingclub_real_training_ready.csv under {processed_dir.resolve()}. "
            f"CSV files: {listing or '(none)'}"
        )
    return hits[0].resolve()


REPO_ROOT = find_repo_root()
DATA_PATH = resolve_training_csv(REPO_ROOT / "datasets" / "processed")

print(f"Resolved CSV: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, low_memory=False)
mem = int(df.memory_usage(deep=True).sum())

print(f"Shape: {df.shape}")
print(f"Memory (deep estimate): {mem / (1024 ** 2):.2f} MiB ({mem:,} bytes)")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

## 2. Split features / target

In [ ]:
TARGET = "default_risk"
if TARGET not in df.columns:
    raise KeyError(f"Missing target column {TARGET!r}")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

print(f"Feature count: {X.shape[1]}")
print("Class distribution (full data):")
print(y.value_counts().sort_index().to_string())
print(y.value_counts(normalize=True).sort_index().round(4).to_string())

## 3. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print("\ny_train distribution:")
print(y_train.value_counts(normalize=True).round(4).sort_index().to_string())
print("\ny_test distribution:")
print(y_test.value_counts(normalize=True).round(4).sort_index().to_string())

## 4. Feature scaling (Logistic Regression only)

`StandardScaler` is **fit on `X_train` only** and applied to train/test for the linear model. **XGBoost** uses the original unscaled matrices.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_test_lr = scaler.transform(X_test)

print("StandardScaler fitted on X_train; X_train_lr / X_test_lr ready for LogisticRegression.")
print("XGBoost will use unscaled X_train / X_test.")

## 5. Train models

**XGBoost:** `scale_pos_weight = (# negative / # positive)` from `y_train` so the minority **risky** class (label `1`) is up-weighted. Other hyperparameters per imbalance fix.

In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError:
    print("Installing xgboost...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "-q"])
    from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
    solver="lbfgs",
)
lr_model.fit(X_train_lr, y_train)

n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
if n_pos == 0:
    raise ValueError("y_train has no positive samples (class 1); cannot set scale_pos_weight.")
scale_pos_weight = n_neg / n_pos
print(
    f"y_train balance: neg={n_neg:,}, pos={n_pos:,}, "
    f"positive_rate={n_pos / (n_neg + n_pos):.4f}"
)
print(f"scale_pos_weight (neg / pos): {scale_pos_weight:.4f}")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)
xgb_model.fit(X_train, y_train)

print("Trained: LogisticRegression (scaled features)")
print("Trained: XGBClassifier (unscaled features, scale_pos_weight applied)")

## 6. Evaluate both models

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def eval_binary(name: str, y_true, y_pred, y_proba) -> dict:
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print(f"\n========== {name} ==========")
    for k, v in out.items():
        print(f"{k:12s}: {v:.4f}")
    print("\nConfusion matrix (rows=true, cols=pred):")
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False)
    ax.set_title(f"{name} — confusion matrix")
    plt.show()
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=4))
    return out


y_pred_lr = lr_model.predict(X_test_lr)
y_proba_lr = lr_model.predict_proba(X_test_lr)[:, 1]
metrics_lr = eval_binary("Logistic Regression", y_test, y_pred_lr, y_proba_lr)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
metrics_xgb = eval_binary("XGBoost (reweighted)", y_test, y_pred_xgb, y_proba_xgb)

comparison = pd.DataFrame(
    {
        "recall": [metrics_lr["recall"], metrics_xgb["recall"]],
        "roc_auc": [metrics_lr["roc_auc"], metrics_xgb["roc_auc"]],
        "f1": [metrics_lr["f1"], metrics_xgb["f1"]],
        "precision": [metrics_lr["precision"], metrics_xgb["precision"]],
        "accuracy": [metrics_lr["accuracy"], metrics_xgb["accuracy"]],
    },
    index=["Logistic Regression", "XGBoost"],
)
print("\n========== Comparison (selection: recall → ROC AUC → F1) ==========")
print(comparison.round(4).to_string())
print("\nNote: accuracy is shown for context only; it is not used to pick the best model.")

## 7. ROC comparison

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, y_proba_lr, name="Logistic Regression", ax=ax)
RocCurveDisplay.from_predictions(y_test, y_proba_xgb, name="XGBoost (scale_pos_weight)", ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="chance")
ax.set_title("ROC curve comparison (test set)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. XGBoost feature importance (top 20)

In [ ]:
feat_imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
print(feat_imp.to_string())

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.sort_values().plot(kind="barh", ax=ax, color="#3498db")
ax.set_title("XGBoost — top 20 feature importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Model selection

In [ ]:
def compare_tuple(m: dict) -> tuple:
    """Lexicographic ranking for default detection: recall first, then ROC AUC, then F1."""
    return (m["recall"], m["roc_auc"], m["f1"])


t_lr = compare_tuple(metrics_lr)
t_xgb = compare_tuple(metrics_xgb)

if t_xgb > t_lr:
    best_name = "xgboost"
    best_model = xgb_model
    choice = "XGBoost wins on (recall, ROC AUC, F1) — `best_credit_default_model.pkl` will hold XGBoost."
elif t_xgb < t_lr:
    best_name = "logistic_regression"
    best_model = lr_model
    choice = "Logistic Regression wins on (recall, ROC AUC, F1) — `best_credit_default_model.pkl` will hold LogisticRegression."
else:
    best_name = "xgboost"
    best_model = xgb_model
    choice = "Tie on (recall, ROC AUC, F1); defaulting to XGBoost."

print("Selection rule: maximize recall, then ROC AUC, then F1 (accuracy ignored for ranking).")
print(f"LogReg tuple (recall, roc_auc, f1): {t_lr}")
print(f"XGBoost tuple (recall, roc_auc, f1): {t_xgb}")
print(f"\nSelected: {best_name}")
print(choice)

## 10. Save artifacts (`joblib`)

In [ ]:
MODELS_DIR = REPO_ROOT / "backend" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

path_best = MODELS_DIR / "best_credit_default_model.pkl"
path_scaler = MODELS_DIR / "logistic_scaler.pkl"
path_features = MODELS_DIR / "feature_columns.pkl"

joblib.dump(best_model, path_best)
joblib.dump(scaler, path_scaler)
joblib.dump(list(X.columns), path_features)

print(f"Saved best model -> {path_best.resolve()}")
print(f"Saved scaler     -> {path_scaler.resolve()}")
print(f"Saved features   -> {path_features.resolve()}")

## 11. Final validation

In [ ]:
print("=== Inference readiness ===")
print(f"Best model type: {best_name}")
print(f"Total feature count: {X.shape[1]}")
print("\nSaved artifacts:")
for p in (path_best, path_scaler, path_features):
    exists = p.is_file()
    size = p.stat().st_size if exists else 0
    print(f"  {p.resolve()}  |  exists={exists}  |  bytes={size:,}")

print("\nBackend inference notes:")
print("  - If serving LogisticRegression: transform new rows with logistic_scaler.pkl then predict with best model if best is LR.")
print("  - If serving XGBoost: predict on raw aligned feature matrix (same column order as feature_columns.pkl).")
print("  - Load feature_columns.pkl to align / validate input schema.")
print("\nReady for backend integration confirmation.")

## 12. Notebook notes

- **Random Forest** is intentionally not trained (per requirements).
- **Logistic Regression** always uses `logistic_scaler.pkl` at inference; **XGBoost** uses raw features aligned to `feature_columns.pkl`.
- **`best_credit_default_model.pkl`** holds whichever estimator won section 9 (ranked by **recall → ROC AUC → F1**).
- **XGBoost** uses `scale_pos_weight = n_negative / n_positive` from `y_train` to mitigate ~13% positive-rate imbalance.
